# 推理 Infra 学习实录 01：从 nano-vLLM 开始

> 归档状态（GitHub 发布副本）：保留 2026-06-15 截图与历史输出记录；本轮未做 GPU 复测。历史截图仍在 [assets](assets/)，数字不因代码修正而改写。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [下一篇](../02/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。


我想用这个系列记录自己学习大模型推理系统的过程。刚开始接触推理 Infra 时，vLLM、KV Cache、prefill、decode、CUDA Graph 这些词很容易让人觉得离自己很远。所以第一篇不急着讲复杂原理，而是选择了 nano-vLLM 这个轻量项目开始学习：先把模型跑起来，再慢慢观察每一步背后发生了什么。

本篇内容会根据下面的流程跑通nano-vllm

1. 看一下当前 GPU 环境
2. 安装需要的依赖
3. 用 ModelScope 下载 Qwen3-0.6B
4. 用 nano-vLLM 生成第一段文本
5. 用 chat template 尝试更正式的聊天格式
6. 跑一次 nano-vLLM benchmark

这篇作为入门笔记：先知道怎么跑、跑出来的结果怎么看，再为后面读源码做准备。

> 如果我现在只会调用模型 API，能不能从一个小项目开始，慢慢理解推理 Infra？

## 1. 为什么从 nano-vLLM 开始

nano-vLLM 是一个轻量级的 vLLM 实现。它保留了大模型推理框架中的核心机制，比如 prefill、decode、KV Cache、请求调度、FlashAttention 和 CUDA Graph，但代码规模更小，更适合用来学习推理系统的基本原理。相比直接阅读生产级 vLLM，nano-vLLM 更像是一个清晰的入门样本，可以帮助我们从“会调用模型 API”逐步过渡到理解推理引擎内部是如何工作的。

本notebook通过第一次跑通nano-vllm，来对一些基本概念有一些了解

- prefill / decode
- KV Cache
- block 管理
- prefix cache
- batch 调度
- CUDA Graph
- FlashAttention

学习顺序：先跑通，再观察吞吐；统一横评见第 07 篇方案。

## 2. 确认环境

大模型推理最常见的问题是环境问题：GPU 不可见、CUDA 不匹配、PyTorch 或 FlashAttention 安装失败。

先看机器和 PyTorch 状态。

In [ ]:
!nvidia-smi

In [ ]:
import sys

print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", e)

## 3. 安装nano-vllm

In [ ]:
%pip install git+https://github.com/GeeeekExplorer/nano-vllm.git@bb823b3e06983d71485a8e1f23715ebd87d98ef8

如果环境里还没有ModelScope，后面下载模型前再安装：

In [ ]:
%pip install modelscope

## 4. 获取 nano-vLLM 源码

也可以选择clone源码，方便后续继续阅读和修改 benchmark。

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("upstream/nano-vllm")
commit = "bb823b3e06983d71485a8e1f23715ebd87d98ef8"
if not repo_dir.exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/GeeeekExplorer/nano-vllm.git", str(repo_dir)],
        check=True,
    )
    subprocess.run(["git", "-C", str(repo_dir), "checkout", commit], check=True)
else:
    print("保留已有 checkout，不自动 pull 或覆盖：", repo_dir)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
).strip()
if actual_commit != commit:
    raise RuntimeError(f"源码版本不符：{actual_commit}；请另选空目录克隆固定版本，不覆盖已有工作。")
print("source commit:", actual_commit)

## 5. 下载小模型

选择 `Qwen/Qwen3-0.6B`，模型小，适合第一次实验。

nano-vLLM 需要本地模型目录，所以用 ModelScope 的 `snapshot_download` 下载到本地。

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)

print("model_dir =", model_dir)

In [ ]:
#检查是否安装完成
import os

print("模型目录:", model_dir)
print("目录内容:")
for name in os.listdir(model_dir)[:30]:
    print(" -", name)

## 6. 第一次推理

先让 nano-vLLM 成功生成文本。

最小调用链：

```text
LLM -> SamplingParams -> prompts -> generate -> outputs
```

脚本里要用 `print` 才会显示结果。

以下三段生成示例各自使用独立、阻塞的 `subprocess.run(..., check=True)`，上一进程结束后才开始下一段。主 Notebook 不保存 LLM 或模型 tensor，避免引擎显存残留与重复初始化默认进程组；若曾执行旧版单元，请先重启 kernel。

In [ ]:
import subprocess
import sys

code = r'''
import sys
model_dir = sys.argv[1]
from nanovllm import LLM, SamplingParams

llm = LLM(
    model_dir,
    enforce_eager=True,
    tensor_parallel_size=1,
)

sampling_params = SamplingParams(
    temperature=0.6,
    max_tokens=256,
)

prompts = ["Hello, Nano-vLLM."]

outputs = llm.generate(prompts, sampling_params)

print(outputs[0]["text"])
'''
subprocess.run([sys.executable, "-c", code, str(model_dir)], check=True)

## 7. 使用 chat template

上一个例子更像文本续写。

Qwen3 这类聊天模型更推荐使用 `apply_chat_template`，把问题包装成 user / assistant 对话格式。

chat 子进程独立创建 eager LLM，不复用上一单元的对象。完整 prefill 后采样首 token，decode 再以该 token 为输入继续生成。

In [ ]:
import subprocess
import sys

code = r'''
import sys
model_dir = sys.argv[1]
from transformers import AutoTokenizer
from nanovllm import LLM, SamplingParams

llm = LLM(model_dir, enforce_eager=True, tensor_parallel_size=1)

tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

sampling_params = SamplingParams(
    temperature=0.6,
    max_tokens=512,
)

raw_prompts = [
    "请用三句话解释什么是大模型推理引擎。"
]

chat_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for p in raw_prompts
]

outputs = llm.generate(chat_prompts, sampling_params)

print(outputs[0]["text"])
'''
subprocess.run([sys.executable, "-c", code, str(model_dir)], check=True)

## 8. nano-vLLM Benchmark

现在测试吞吐。

这里使用随机 token，因此测试的是推理吞吐、调度和 KV Cache 管理，不是回答质量。

默认参数：

```text
num_seqs = 256
max_input_len = 1024
max_output_len = 1024
max_model_len = 4096
```

显存不足时，可以先调小这些值。

benchmark 子进程独立创建 CUDA Graph LLM；预热后先同步 CUDA，用 `perf_counter()` 计时并在结束时同步，按实际输出 `token_ids` 计数。历史截图为旧代码记录，不代表修正代码已复测。

In [ ]:
import subprocess
import sys

code = r'''
import sys
model_dir = sys.argv[1]
from time import perf_counter
import torch
from random import randint, seed
from nanovllm import LLM, SamplingParams

seed(0)

num_seqs = 256
max_input_len = 1024
max_output_len = 1024

llm = LLM(
    model_dir,
    enforce_eager=False,
    max_model_len=4096,
    tensor_parallel_size=1,
)

prompt_token_ids = [
    [randint(0, 10000) for _ in range(randint(100, max_input_len))]
    for _ in range(num_seqs)
]

sampling_params = [
    SamplingParams(
        temperature=0.6,
        ignore_eos=True,
        max_tokens=randint(100, max_output_len),
    )
    for _ in range(num_seqs)
]

llm.generate(["Benchmark: "], SamplingParams())

torch.cuda.synchronize()
t = perf_counter()
outputs = llm.generate(prompt_token_ids, sampling_params, use_tqdm=False)
torch.cuda.synchronize()
elapsed = perf_counter() - t

total_tokens = sum(len(output["token_ids"]) for output in outputs)
throughput = total_tokens / elapsed

nano_result = {
    "engine": "nano-vLLM",
    "total_tokens": total_tokens,
    "time": elapsed,
    "throughput": throughput,
}

print(nano_result)
'''
subprocess.run([sys.executable, "-c", code, str(model_dir)], check=True)

9. 如何解读 benchmark
这个 benchmark 使用随机 token，主要测试：
推理吞吐
batch 调度
KV Cache 管理
decode 性能
CUDA Graph 路径




真实聊天效果要用自然语言 prompt 和 apply_chat_template。